# LTX-Video Mini - Free Google Colab UI

A prompt-to-video Gradio UI using **LTX-Video v0.9.7-distilled** - the predecessor to LTX-2 from the same team (Lightricks). It is a much smaller model (~6 GB instead of 22 GB) so it fits comfortably on free Colab T4.

Trade-offs vs LTX-2: lower visual quality, no audio, but **runs free on Colab** with no /tmp tricks, no gated-model logins, no quantization juggling. Just works.

## How to use

1. Open this notebook in Colab.
2. Runtime > Change runtime type > T4 GPU (free).
3. Runtime > Run all.
4. The last cell prints a public https://*.gradio.live URL - open it.

First boot takes ~5 min (downloads ~6 GB). Each video then takes 30-60 seconds.

## 1. GPU sanity check

In [ ]:
!nvidia-smi || echo 'No GPU. Runtime > Change runtime type > T4 GPU.'
!df -h /content

## 2. Install dependencies (~2 min)

In [ ]:
# diffusers >= 0.32 ships LTXPipeline. Use Colab preinstalled torch.
!pip install -q --upgrade 'diffusers>=0.32' 'transformers>=4.45' 'accelerate>=1.0' 'gradio>=4.44' sentencepiece imageio imageio-ffmpeg
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 3. Write the Gradio app file

We write to `/content/ltx_mini_app.py` so Gradio runs in its own process (avoids cell-output buffering hangs).

In [ ]:
%%writefile /content/ltx_mini_app.py
"""LTX-Video distilled prompt-to-video UI for free Colab T4."""
import os
import time
from pathlib import Path

import gradio as gr
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

MODEL_ID = 'Lightricks/LTX-Video-0.9.7-distilled'
_PIPE = None
OUT_DIR = Path('/content/outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)


def get_pipeline():
    global _PIPE
    if _PIPE is not None:
        return _PIPE
    print('Loading LTX-Video pipeline (first time downloads ~6 GB)...')
    pipe = LTXPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
    pipe.enable_model_cpu_offload()
    _PIPE = pipe
    return pipe


def generate(prompt, negative_prompt, width, height, num_frames, num_inference_steps, fps, seed,
             progress=gr.Progress(track_tqdm=True)):
    if not prompt or not prompt.strip():
        raise gr.Error('Please enter a prompt.')
    progress(0.0, desc='Loading pipeline...')
    pipe = get_pipeline()
    width = max(256, (int(width) // 32) * 32)
    height = max(256, (int(height) // 32) * 32)
    nf = int(num_frames)
    if (nf - 1) % 8 != 0:
        nf = ((nf - 1) // 8) * 8 + 1
    nf = max(nf, 9)
    progress(0.1, desc='Running diffusion...')
    gen = torch.Generator(device='cuda').manual_seed(int(seed))
    out = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt or 'worst quality, blurry, distorted, low resolution',
        width=width,
        height=height,
        num_frames=nf,
        num_inference_steps=int(num_inference_steps),
        generator=gen,
    )
    video_frames = out.frames[0]
    out_path = str(OUT_DIR / ('ltxv_' + str(int(time.time())) + '_' + str(int(seed)) + '.mp4'))
    progress(0.95, desc='Encoding video...')
    export_to_video(video_frames, out_path, fps=int(fps))
    progress(1.0, desc='Done')
    return out_path, 'Saved to ' + out_path


PROMPT_HINT = (
    'A cinematic shot of a fox running through a snowy forest at dawn, '
    'soft golden light filtering through the trees, smooth tracking camera, '
    'shallow depth of field.'
)


with gr.Blocks(title='LTX-Video Mini') as demo:
    gr.Markdown('# LTX-Video Mini\nRunning **' + MODEL_ID + '** on Colab T4 with model_cpu_offload. ~30-60 sec/video at 768x512x49 frames.')
    with gr.Row():
        with gr.Column(scale=3):
            prompt = gr.Textbox(label='Prompt', lines=4, placeholder=PROMPT_HINT)
            neg = gr.Textbox(label='Negative Prompt', value='worst quality, blurry, distorted, low resolution', lines=2)
            with gr.Accordion('Advanced settings', open=False):
                with gr.Row():
                    width = gr.Slider(256, 1024, value=768, step=32, label='Width')
                    height = gr.Slider(256, 1024, value=512, step=32, label='Height')
                with gr.Row():
                    num_frames = gr.Slider(9, 161, value=49, step=8, label='Frames (8k+1)')
                    fps = gr.Slider(8, 30, value=24, step=1, label='FPS')
                with gr.Row():
                    steps = gr.Slider(4, 20, value=8, step=1, label='Inference steps (distilled = ~8)')
                    seed = gr.Number(value=42, precision=0, label='Seed')
            btn = gr.Button('Generate Video', variant='primary')
        with gr.Column(scale=2):
            out_video = gr.Video(label='Result', autoplay=True)
            status = gr.Markdown('')
    gr.Examples(
        examples=[
            ['A serene aerial shot drifting over a turquoise lagoon at sunset, palm trees swaying in the warm golden light.'],
            ['A close-up of a hummingbird hovering near a bright red flower, slow motion, soft natural backlight.'],
            ['A neon-lit cyberpunk alley at night, rain on the asphalt reflecting the signs, a lone figure walking away from camera.'],
        ],
        inputs=[prompt],
    )
    btn.click(generate, [prompt, neg, width, height, num_frames, steps, fps, seed], [out_video, status])

demo.queue().launch(share=True, server_name='0.0.0.0', server_port=7860)


## 4. Run the app and get the public URL

In [ ]:
!python /content/ltx_mini_app.py

## Tips

- Distilled variant uses 8 inference steps by default. Bumping it does not help much.
- Default 768x512x49 frames is ~2 sec at 24 fps. Push to 768x512x97 for ~4 sec; higher only if VRAM allows.
- First generation: ~90 sec (model load + first run). Subsequent: ~30-60 sec.
- Generated MP4s land in /content/outputs/ - visible in Colab left-side Files panel.
- Free Colab disconnects after ~12h or 90min idle. Re-run the last cell to relaunch (HF cache survives).
- This is NOT LTX-2 - it is the older v0.9.7. Quality is good but not LTX-2 level.